# BirdCLEF+ 2026 — EDA

Pantanal soundscape bird species identification.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

DATA = Path(os.getenv("KEGO_PATH_DATA", "data")) / "birdclef" / "birdclef-2026"
print("Data dir:", DATA)
print("Exists:", DATA.exists())

## 1. Data structure

In [ ]:
# List top-level files and directories
for p in sorted(DATA.iterdir()):
    if p.is_dir():
        n = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"  {p.name}/  ({n} files)")
    else:
        print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

## 2. Training metadata

In [ ]:
# Key facts discovered:
# - File is train.csv (not train_metadata.csv)
# - primary_label = mixed format: numeric iNat taxon ID (non-birds) or eBird code (birds)
# - filename includes subdirectory: e.g. "64898/XC12345.ogg"
# - taxonomy.csv has 234 species (all submission targets); only 206 appear in train.csv
# - 28 species are zero-shot (in taxonomy but no training data — mostly sonotype splits)
meta = pd.read_csv(DATA / "train.csv")
taxonomy = pd.read_csv(DATA / "taxonomy.csv")
print("Train recordings:", len(meta), "| Taxonomy species:", len(taxonomy))
meta.head()

In [ ]:
print("Columns:", meta.columns.tolist())
print("\nDtypes:")
print(meta.dtypes)
print("\nNull counts:")
print(meta.isnull().sum())

In [ ]:
# Species distribution
species_col = "primary_label" if "primary_label" in meta.columns else meta.columns[0]
n_species = meta[species_col].nunique()
counts = meta[species_col].value_counts()

print(f"Total recordings: {len(meta):,}")
print(f"Unique species: {n_species}")
print(f"\nRecordings per species — min: {counts.min()}, median: {counts.median():.0f}, max: {counts.max()}")
print(f"Species with <10 recordings: {(counts < 10).sum()}")
print(f"Species with <5 recordings: {(counts < 5).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Per-species count distribution
axes[0].hist(counts.values, bins=50, log=True)
axes[0].set_xlabel("Recordings per species")
axes[0].set_ylabel("Count (log)")
axes[0].set_title("Species recording distribution")

# Top 30 species
counts.head(30).plot(kind="barh", ax=axes[1])
axes[1].set_title("Top 30 species by recording count")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Zero-shot species: in taxonomy but NOT in training data
train_labels = set(meta["primary_label"].astype(str))
tax_labels = set(taxonomy["primary_label"].astype(str))
zero_shot = tax_labels - train_labels
print(f"Zero-shot species (taxonomy only, no training data): {len(zero_shot)}")
taxonomy[taxonomy["primary_label"].astype(str).isin(zero_shot)][
    ["primary_label", "scientific_name", "common_name", "class_name"]
]

## 3. Secondary labels

In [ ]:
if "secondary_labels" in meta.columns:
    has_secondary = meta["secondary_labels"].notna() & (meta["secondary_labels"] != "[]")
    print(f"Recordings with secondary labels: {has_secondary.sum()} ({has_secondary.mean():.1%})")
    # Parse and count unique secondary species
    import ast

    secondary = meta.loc[has_secondary, "secondary_labels"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else []
    )
    flat = [s for labels in secondary for s in labels]
    print(f"Total secondary label occurrences: {len(flat)}")
    print(f"Unique species appearing as secondary: {len(set(flat))}")
else:
    print("No secondary_labels column found")

## 4. Audio properties

In [ ]:
# Sample a few audio files to check format
train_audio_dir = DATA / "train_audio"
sample_files = list(train_audio_dir.rglob("*.ogg"))[:10] + list(train_audio_dir.rglob("*.wav"))[:10]
sample_files = sample_files[:10]

print(f"Found {len(list(train_audio_dir.rglob('*.*')))} total audio files")
print("\nSample file properties:")
durations = []
for f in sample_files:
    info = sf.info(f)
    durations.append(info.duration)
    print(f"  {f.name}: {info.samplerate}Hz, {info.duration:.1f}s, {info.channels}ch")

print(f"\nDuration stats — min: {min(durations):.1f}s, median: {np.median(durations):.1f}s, max: {max(durations):.1f}s")

## 5. Mel-spectrogram visualization

In [ ]:
SR = 32000
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 1024
FMIN = 20
FMAX = 16000
CLIP_DURATION = 5  # seconds


def load_clip(path, sr=SR, duration=CLIP_DURATION, offset=0):
    y, _ = librosa.load(path, sr=sr, duration=duration, offset=offset, mono=True)
    if len(y) < sr * duration:
        y = np.pad(y, (0, sr * duration - len(y)))
    return y


def make_melspec(y, sr=SR):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
        n_fft=N_FFT,
        fmin=FMIN,
        fmax=FMAX,
    )
    return librosa.power_to_db(mel, ref=np.max)


# Plot 6 random training clips
fig, axes = plt.subplots(2, 3, figsize=(18, 6))
for ax, f in zip(axes.flat, sample_files[:6]):
    y = load_clip(f)
    spec = make_melspec(y)
    librosa.display.specshow(
        spec,
        sr=SR,
        hop_length=HOP_LENGTH,
        fmin=FMIN,
        fmax=FMAX,
        x_axis="time",
        y_axis="mel",
        ax=ax,
    )
    species = f.parent.name
    ax.set_title(species, fontsize=9)
    ax.set_xlabel("")
plt.suptitle("Sample mel-spectrograms (5s clips)", y=1.01)
plt.tight_layout()
plt.show()

## 6. Test / soundscape data

In [ ]:
# Check test soundscapes
for test_dir in ["test_soundscapes", "unlabeled_soundscapes"]:
    d = DATA / test_dir
    if d.exists():
        files = list(d.rglob("*.*"))
        print(f"{test_dir}/: {len(files)} files")
        if files:
            info = sf.info(files[0])
            print(f"  Sample: {info.samplerate}Hz, {info.duration:.1f}s")
    else:
        print(f"{test_dir}/: not found")

In [ ]:
# Check sample submission
for name in ["sample_submission.csv", "taxonomy.csv"]:
    p = DATA / name
    if p.exists():
        df = pd.read_csv(p)
        print(f"{name}: {df.shape}")
        print(df.head(3))
        print()